# __MODEL_LABEL_MARKDOWN__: data ingestion

This notebook is the only governed step that creates or replaces the
final model-frame handoff. Keep the source query and every transform
visible, then save the verified artifact for model training.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

DATA_AS_OF = ""  # Required ISO date: the dataset version, not a deployment date.
REPLACE_MODEL_FRAME = False


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import numpy as np
import pandas as pd

from pricing_pipeline.notebook import connect, save_model_frame

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"


## Connect and verify the source/audit destination


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


## Read source data

Replace this demo with the normal work query. Keep source-owned keys,
target, exposure/weights, split fields, and the data-as-of value.


In [ ]:
if not DATA_AS_OF.strip():
    raise ValueError("Set the required DATA_AS_OF dataset version stamp.")
rng = np.random.default_rng(42)
raw = pd.DataFrame({
    "__PRIMARY_KEY__": np.arange(1, 101),
    "__FEATURE_NAME__": rng.normal(size=100),
    "segment": rng.choice(["A", "B", "C"], size=100),
    "data_as_of": [DATA_AS_OF] * 100,
})
raw["__TARGET_NAME__"] = rng.poisson(
    np.exp(
        -0.5
        + 0.25 * raw["__FEATURE_NAME__"]
        + raw["segment"].map({"A": 0.0, "B": 0.2, "C": -0.1})
    )
)
display({"Rows loaded": len(raw), "Columns loaded": len(raw.columns)})


## Build the final model frame

Apply production-intended feature transforms here. Preserve deterministic
row ordering and retain only the columns needed by the training spec.


In [ ]:
frame = (
    raw.loc[
        :,
        [
            "__PRIMARY_KEY__",
            "__TARGET_NAME__",
            "__FEATURE_NAME__",
            "segment",
            "data_as_of",
        ],
    ]
    .sort_values("__PRIMARY_KEY__")
    .reset_index(drop=True)
)
display(frame.head())


## Save the verified notebook handoff


In [ ]:
frame_artifact = save_model_frame(
    frame,
    FRAME_ARTIFACT_PATH,
    replace=REPLACE_MODEL_FRAME,
)
display(frame_artifact)
